In [32]:
import pandas as pd
#시군구별 계약종별별 전력사용량	((단위 : kWh)) 불필요 열 제거
elect = pd.read_csv('C:/skn24/2601_mini_team/EDA_MINI_1TEAM/jaehoon/미니프로젝트/data/elect_2015_2024.csv',encoding='CP949')
elect['시도'].unique()


# 강원도만 뽑기 
chung_region = ['충청북도','충청남도','세종특별자치시','대전광역시']
chung_elect = elect[elect['시도'].isin(chung_region)]
chung_elect['시도'].unique()


array(['대전광역시', '충청북도', '충청남도', '세종특별자치시'], dtype=object)

In [33]:
chung_elect[chung_elect['시도']=='세종특별자치시']

,Unnamed: 0,연도,시도,시군구,계약종별,1월,2월,3월,4월,5월,6월,7월,8월,9월,10월,11월,12월
1825,1825,2017,세종특별자치시,세종시,주택용,30785254,32963495,28787637,30228547,28632317,29183239,32930891,41960581,35566246,30786773,32839328,35711477
1826,1826,2017,세종특별자치시,세종시,일반용,49725675,50318535,44461667,39910314,36836117,40865391,48438586,53169192,45945230,38246642,41377065,51963283
1827,1827,2017,세종특별자치시,세종시,교육용,6981435,6621509,6772507,6303004,4389522,4721145,5246661,5332800,5223876,4446558,5305997,8184737
1828,1828,2017,세종특별자치시,세종시,산업용,144986473,139535701,146708789,141586168,140039222,147684714,151062808,144425525,148812979,131735221,143461288,146490821
1829,1829,2017,세종특별자치시,세종시,농사용,6240290,6820913,5828650,4876625,4461235,5399740,5706990,7047145,6021411,4889942,8832339,6346428
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18351,18351,2024,세종특별자치시,세종시,산업용,184998667,166932034,176182630,170626917,177685228,180889407,185779291,184223308,176158729,177576447,173938421,180806812
18352,18352,2024,세종특별자치시,세종시,농사용,8478103,8515706,7451538,6592303,6205161,7701675,8124793,10767466,10907512,7744199,9378854,6824009
18353,18353,2024,세종특별자치시,세종시,가로등,3508720,3332958,3125423,3029530,2795076,2703428,2616324,2775356,2944976,3170166,3435927,3572786
18354,18354,2024,세종특별자치시,세종시,심 야,10025377,9874870,8053238,5659135,2972642,2291505,1846149,1767560,1592048,1958635,3687815,7148324


In [34]:
#기상청 데이터와 일치시키기 
chung_elect['지점명'] = chung_elect['시군구'].str.replace('시|군|구','',regex=True)
chung_elect = chung_elect.rename(columns={'연도':'years'})

# 기상청 데이터의 강원도 지역과 동일한 한국전력의 측정 지역 추출 (기상청 강원도 데이터의 지역수(9개 지역)가 더 적기 때문에 기준으로 진행)
chung_temp_list = [
    "천안", "아산", "서산", "당진", "공주", "보령", "논산", "계룡",
    "홍성", "예산", "태안", "금산", "부여", "서천", "청양", "청주", "충주", "제천",
    "보은", "옥천", "영동", "괴산", "음성", "단양", "증평", "진천","대전", '세종'
]

chung_elect.loc[chung_elect['시도'] == '대전광역시', '시군구'] = (
    chung_elect.loc[chung_elect['시도'] == '대전광역시', '시군구']
        .str.replace('^(동구|중구|서구|대덕구|유성구)$', '대전', regex=True)
)


chung_elect_list = []
for i in chung_elect['시군구'].drop_duplicates():
    if i in chung_elect_list: # 기상청 강원도 리스트
        chung_elect_list.append(i)
chung_elect_list

# 강원도 지역의 속한 데이터프레임 뽑기 
chung_keyword = "|".join(chung_elect_list)
chung_elect = chung_elect[chung_elect['시군구'].str.contains(chung_keyword)]

# # 2번째 방법
# chung_elect = chung_elect[chung_elect['계약종별']=='합 계'].reset_index(drop=True)
# chung_elect
chung_elect['계약종별'].unique()




C:\Users\Playdata\AppData\Local\Temp\ipykernel_12612\4005175435.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  chung_elect['지점명'] = chung_elect['시군구'].str.replace('시|군|구','',regex=True)


array(['주택용', '일반용', '교육용', '산업용', '농사용', '가로등', '심 야', '합 계'],
      dtype=object)

In [35]:
contract_types = ['주택용', '일반용', '교육용', '산업용', '농사용', '가로등', '심 야', '합 계']

chung_dfs = {}

for ct in contract_types:
    chung_dfs[ct] = (
        chung_elect[chung_elect['계약종별'] == ct]
        .reset_index(drop=True)
    )
gangwon_house = chung_dfs['주택용']
gangwon_industry = chung_dfs['산업용']
gangwon_edu = chung_dfs['교육용']
gangwon_farm = chung_dfs['농사용']
gangwon_light = chung_dfs['가로등']
gangwon_night = chung_dfs['심 야']
gangwon_total = chung_dfs['합 계']
gangwon_total


,Unnamed: 0,years,시도,시군구,계약종별,1월,2월,3월,4월,5월,6월,7월,8월,9월,10월,11월,12월,지점명
0,518,2017,대전광역시,대전,합 계,86569098,90065720,75843798,72708852,62739965,65170853,73656218,83886071,71786992,62228385,66785018,83345265,동
1,526,2017,대전광역시,대전,합 계,94753039,94893070,79375443,75791824,68014602,72523755,85326835,94002343,77819631,68002315,65981578,87408676,중
2,534,2017,대전광역시,대전,합 계,168864617,173218892,150343864,145308456,130125768,137045409,156411826,184345486,158231933,130504198,142074291,161900146,서
3,542,2017,대전광역시,대전,합 계,241275329,237067809,238926866,229454148,219456542,229241205,237762737,248953096,235462816,205569437,225126346,232828118,대덕
4,550,2017,대전광역시,대전,합 계,257067038,253405068,234896686,225690707,220010703,237239968,273222981,284300729,244201736,214358745,230926437,270304579,유성
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
319,17699,2024,충청남도,홍성군,합 계,107913553,103789076,94539935,86641160,76865445,78531656,88927681,108494199,104924754,87305722,94730107,93794622,홍성
320,17707,2024,충청남도,예산군,합 계,131481351,123049011,118800040,112819104,105726196,105634750,114086072,125734793,122493610,110641742,115108174,121103055,예산
321,17715,2024,충청남도,당진시,합 계,662147182,554399766,593746355,565837256,574830994,572127372,575528585,593243944,505593252,468181313,507749074,545077410,당진
322,17723,2024,충청남도,태안군,합 계,80551338,82210966,92930999,72398048,63543907,83127554,80274588,73611767,90904969,101341990,103961479,78493279,태안


In [36]:
for contract_type, df in chung_dfs.items():
    melted_df = pd.melt(
        df,
        id_vars=['years', '시도', '시군구', '계약종별', '지점명'],
        value_vars=['1월','2월','3월','4월','5월','6월',
                    '7월','8월','9월','10월','11월','12월'],
        var_name='month',
        value_name='전력량'
    )

    melted_df['month'] = melted_df['month'].str.replace('월','', regex=True).astype(int)
    melted_df['전력량'] = melted_df['전력량'].astype(int)

    melted_df.to_csv(
        f'C:/skn24/2601_mini_team/EDA_MINI_1TEAM/jaehoon/미니프로젝트/data/preprocessing/충청도 데이터/{contract_type}_total_elect.csv',
        index=False,
        encoding='CP949'
    )

In [37]:
melted_df['시도'].unique()

array(['대전광역시', '충청북도', '충청남도', '세종특별자치시'], dtype=object)